# Study of the stream mask: the known streams of a footprint

The known streams of the footprint, listed by hand in `data/known_streams/<data name>_<subset>.csv`
(`galstreams_catalog.load_known_streams`): `stream_name`, `track_name` (galstreams), and two columns:
- `status`, `masked` or `not_masked`: whether the stream mask of a candidate investigation (`CandidateInvestigation`)
  may remove it;
- `observed`, `yes`, `no` or `unclear`: whether the stream is seen in the matched-filter maps.

How they are masked in a candidate investigation: the `masked` streams not wider than `STREAM_MASK_MAX_WIDTH_DEG`
whose distance modulus comes within `STREAM_MASK_MAX_DM_DIFFERENCE` of that of the candidate, and those given in
`always_masked_streams` whatever their distance modulus (e.g. the `observed` ones, `observed_known_streams()`, whose
stars could leak into the Hess diagrams of the candidate).

This notebook reads the file and shows its streams, to fill `observed` and `status` (in the CSV file, by hand): the
matched-filter map of the whole footprint over distance moduli with every stream of the file (a GIF), and for each
stream zoomed maps around its distance modulus.

Steps:
1. inputs;
2. stars, footprint and the search, prepared once;
3. the streams of the file, and their galstreams tracks;
4. the maps of the whole footprint over distance moduli, with the streams coloured by status, and the GIF;
5. each stream, zoomed, at and around its distance modulus.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from pathlib import Path

import healpy as hp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from matplotlib.lines import Line2D
from matplotlib.patheffects import withStroke

from streameucliddes.analysis.build_masks import (
    STREAM_MASK_MAX_DM_DIFFERENCE,
    STREAM_MASK_MAX_WIDTH_DEG,
    galstreams_mask_tracks,
)
from streameucliddes.analysis.cmd_match_filter import get_match_filter
from streameucliddes.analysis.generic_stream_search import (
    generic_search_dm_scan,
    generic_search_single_dm,
    prepare_generic_search,
)
from streameucliddes.catalogs.euclid_catalog import default_star_cuts, get_euclid_stars
from streameucliddes.catalogs.isochrone_parameters import get_isochrone
from streameucliddes.catalogs.galstreams_catalog import MASKED_STATUS, known_streams_path, load_known_streams
from streameucliddes.catalogs.survey_footprint import in_healpix_mask, survey_footprint_healpix_mask
from streameucliddes.plotting.make_videos import make_gif_from_plots
from streameucliddes.plotting.plot_generic_search import DENSITY_PERCENTILES, plot_generic_search_residual_map
from streameucliddes.plotting.plot_stream_frame_search import draw_stream_track_on_sky
from streameucliddes.plotting.plot_style import CATEGORICAL_COLORS, format_distance_modulus

pd.set_option("display.max_colwidth", None)

## 1. Inputs

The data, the star sample, the match filter and the isochrone as in `find_candidates`. `DISTANCE_MODULI`: one frame
of the whole footprint each. `ZOOM_DM_OFFSETS`: the zoomed maps of a stream, at its distance modulus plus these;
`ZOOM_MARGIN_DEG` around its part in the footprint. The frames, the GIF and the zooms go to
`figures/outputs/study_mask/` (gitignored).

In [ ]:
euclid_release = "dr1"
des = False
data_name = f"euclid_{euclid_release}" + ("_des" if des else "")
subset = "south"  # the south footprint; "northwest", "northeast" later

star_selection = "morphology_cuts"  # or "xgboost_classifier" (see default_star_cuts)
star_cuts = default_star_cuts(data_name, star_selection)

MATCH_FILTER = get_match_filter()  # the default: Euclid DR1 VIS vs VIS - H
ISOCHRONE = get_isochrone()  # the default: 13 Gyr, [Fe/H] = -1.9 in the default isochrone model
NSIDE = 512  # pixels of about 0.11 deg
SMOOTHING_DEG = 0.3
DISTANCE_MODULI = np.round(np.arange(14.0, 19.5 + 0.125, 0.25), 2)  # one frame of the whole footprint each
N_WORKERS = 1  # maps computed in parallel (forked processes sharing the stars)
ZOOM_DM_OFFSETS = (-0.5, 0.0, 0.5)  # zoomed maps of a stream: at its distance modulus plus these
ZOOM_MARGIN_DEG = 4.0

# The streams on the maps: colour by status; solid within STREAM_MASK_MAX_DM_DIFFERENCE of the distance modulus of
# the map (where a candidate there would mask them), dotted beyond
STATUS_COLORS = {"masked": CATEGORICAL_COLORS[4], "not_masked": CATEGORICAL_COLORS[1]}
NEAR_STYLE = {"lw": 1.6, "ls": "-", "alpha": 0.9}
FAR_STYLE = {"lw": 1.0, "ls": ":", "alpha": 0.5}

OUTPUT_FOLDER = Path.cwd().parent / "figures" / "outputs" / "study_mask"  # run from notebooks/
FRAME_DPI = 100
FRAME_DURATION_MS = 600

## 2. Stars, footprint and the search, prepared once

In [ ]:
stars = get_euclid_stars(data_name, cuts=star_cuts, subset=subset)
footprint = survey_footprint_healpix_mask(stars["ra"].to_numpy(), stars["dec"].to_numpy(), NSIDE, nest=False)
search = prepare_generic_search(stars, match_filter=MATCH_FILTER, nside=NSIDE, footprint_mask=footprint, nest=False)
print(f"{len(stars):,} stars, {search.keep.sum():,} kept")

## 3. The streams of the file, and their tracks

The file of the footprint (`known_streams_path`), and the galstreams track of each stream with its distance moduli in
the footprint and its width, as the stream mask takes them (`galstreams_mask_tracks(..., never_masked=())`). `narrow`:
not wider than `STREAM_MASK_MAX_WIDTH_DEG` (wider ones are never masked). Printed: the streams crossing the footprint
missing from the file (add a row by hand), and the rows whose track does not cross it (left out).

In [ ]:
path = known_streams_path(f"{data_name}_{subset}")
table = load_known_streams(path)
crossing = {track["name"]: track for track in galstreams_mask_tracks(footprint, nest=False, never_masked=())}
print(path)
print("Crossing the footprint, not in the file:",
      [name for name in crossing if name not in set(table["track_name"])] or "none")
print("In the file, not crossing the footprint:",
      [name for name in table["track_name"] if name not in crossing] or "none")

known_streams = table[table["track_name"].isin(list(crossing))].reset_index(drop=True)
tracks = {row.stream_name: crossing[row.track_name] for row in known_streams.itertuples()}
for column in ("dm_min", "dm_max", "width_deg"):
    known_streams[column] = [tracks[name][column] for name in known_streams["stream_name"]]
known_streams["narrow"] = ~(known_streams["width_deg"] > STREAM_MASK_MAX_WIDTH_DEG)  # NaN: unknown, narrow
known_streams.round(2)

## 4. The whole footprint over distance moduli, and the GIF

At each distance modulus, two panels of the same corrected matched-filter map (`plot_generic_search_residual_map`,
its own colour scale): alone, to see the streams; and with every stream of the list, coloured by status
(`STATUS_COLORS`), solid within `STREAM_MASK_MAX_DM_DIFFERENCE` of the distance modulus of the map, dotted beyond.
The title names those a candidate at this distance modulus would mask (`masked`, narrow and near). The
frames are `dm<dm>.png` in `OUTPUT_FOLDER / "frames"`, the GIF `known_streams_dm_scan.gif` in `OUTPUT_FOLDER`, from
the frames of this run only.

In [ ]:
footprint_lon, _ = hp.pix2ang(NSIDE, np.flatnonzero(~footprint), lonlat=True)
LON_CENTRE = float(np.degrees(np.angle(np.mean(np.exp(1j * np.radians(footprint_lon))))))  # circular mean


def lon_around_centre(lon):
    """Longitudes within 180 deg of the footprint centre (no jump at ra = 0)."""
    return (np.asarray(lon) - LON_CENTRE + 180.0) % 360.0 - 180.0 + LON_CENTRE


def part_in_footprint(stream_name):
    """(ra, dec) of the points of the track of a stream inside the footprint."""
    ra, dec = np.asarray(tracks[stream_name]["ra"], float), np.asarray(tracks[stream_name]["dec"], float)
    inside = ~in_healpix_mask(ra, dec, footprint, nest=False)
    return ra[inside], dec[inside]


def is_near(row, distance_modulus):
    """Whether the distance modulus of the stream comes within STREAM_MASK_MAX_DM_DIFFERENCE of this one."""
    return (row.dm_min - STREAM_MASK_MAX_DM_DIFFERENCE <= distance_modulus
            <= row.dm_max + STREAM_MASK_MAX_DM_DIFFERENCE)


def draw_known_streams(ax, distance_modulus):
    """The streams of the list on a sky map, coloured by status, named; solid when near, dotted beyond."""
    for row in known_streams.itertuples():
        track = tracks[row.stream_name]
        color = STATUS_COLORS[row.status]
        draw_stream_track_on_sky(ax, track["ra"], track["dec"], color=color,
                                 **(NEAR_STYLE if is_near(row, distance_modulus) else FAR_STYLE))
        ra, dec = part_in_footprint(row.stream_name)
        if ra.size:
            middle = ra.size // 2
            ax.text(ra[middle], dec[middle], f" {row.stream_name}", color=color, fontsize=8, ha="left",
                    va="bottom", clip_on=True, path_effects=[withStroke(linewidth=2.0, foreground="white")])
    handles = [Line2D([], [], color=color, label=status, **NEAR_STYLE) for status, color in STATUS_COLORS.items()]
    handles.append(Line2D([], [], color="0.4", label=f"beyond {STREAM_MASK_MAX_DM_DIFFERENCE:g} mag", **FAR_STYLE))
    ax.legend(handles=handles, loc="upper right", fontsize="small", framealpha=0.8)


def plot_footprint_frame(result, distance_modulus):
    """The map alone, and with the streams of the list."""
    fig, (ax_map, ax_streams) = plt.subplots(1, 2, figsize=(20.0, 7.5))
    plot_generic_search_residual_map(result, title="corrected map", ax=ax_map)
    _, ax_streams = plot_generic_search_residual_map(result, title="known streams, by status", ax=ax_streams)
    draw_known_streams(ax_streams, distance_modulus)
    masked = [row.stream_name for row in known_streams.itertuples()
              if row.status == MASKED_STATUS and row.narrow and is_near(row, distance_modulus)]
    fig.suptitle(f"dm {format_distance_modulus(distance_modulus)} ({result['distance_kpc']:.1f} kpc); masked by a "
                 f"candidate here: {', '.join(masked) if masked else 'none'}")
    return fig

In [ ]:
%%time
frames_folder = OUTPUT_FOLDER / "frames"
frames_folder.mkdir(parents=True, exist_ok=True)
frame_paths = []
for dm, result in generic_search_dm_scan(search=search, distance_moduli=DISTANCE_MODULI, isochrone=ISOCHRONE,
                                         smoothing_deg=SMOOTHING_DEG, n_workers=N_WORKERS):
    fig = plot_footprint_frame(result, dm)
    frame_paths.append(frames_folder / f"dm{dm:.2f}.png")
    fig.savefig(frame_paths[-1], dpi=FRAME_DPI)
    plt.close(fig)
gif_path = OUTPUT_FOLDER / "known_streams_dm_scan.gif"
make_gif_from_plots(gif_path, frame_paths, duration=FRAME_DURATION_MS)
print(f"{len(frame_paths)} frames in {frames_folder}, GIF {gif_path}")

In [ ]:
Image(filename=gif_path)

## 5. Each stream, zoomed, at and around its distance modulus

For each stream of the list, the maps around its part in the footprint (`ZOOM_MARGIN_DEG`) at its distance modulus
(middle of `dm_min` and `dm_max`) plus `ZOOM_DM_OFFSETS`, alone, then the one at its distance modulus with its track
(coloured by status). The colour scale of each zoom is computed on its own pixels (`DENSITY_PERCENTILES`), not on the
whole footprint: a dense patch elsewhere does not wash it out. One figure per stream in `OUTPUT_FOLDER / "streams"`.
`STREAMS_SHOWN`: a few names to look at only them (None: all).

In [ ]:
STREAMS_SHOWN = None  # e.g. ["Phoenix", "Turbio"]

pixel_lon, pixel_lat = hp.pix2ang(NSIDE, np.arange(hp.nside2npix(NSIDE)), lonlat=True)
pixel_lon = lon_around_centre(pixel_lon)


def zoom_colour_scale(result, lon_range, lat_range):
    """(vmin, vmax): the DENSITY_PERCENTILES of the unmasked pixels of the zoom."""
    residual = np.ma.asarray(result["residual"])
    in_zoom = ((pixel_lon >= lon_range[0]) & (pixel_lon <= lon_range[1]) & (pixel_lat >= lat_range[0])
               & (pixel_lat <= lat_range[1]) & ~np.ma.getmaskarray(residual))
    values = np.asarray(residual[in_zoom])
    return tuple(np.percentile(values, DENSITY_PERCENTILES)) if values.size else (None, None)


def plot_stream_zooms(row):
    """The maps of a stream at its distance modulus plus ZOOM_DM_OFFSETS, then with its track."""
    ra, dec = part_in_footprint(row.stream_name)
    lon = lon_around_centre(ra)
    lon_range = (lon.min() - ZOOM_MARGIN_DEG, lon.max() + ZOOM_MARGIN_DEG)
    lat_range = (max(dec.min() - ZOOM_MARGIN_DEG, -89.0), min(dec.max() + ZOOM_MARGIN_DEG, 89.0))
    stream_dm = round(0.5 * (row.dm_min + row.dm_max), 2)
    offsets = (*ZOOM_DM_OFFSETS, 0.0)  # the last: with the track
    fig, axes = plt.subplots(1, len(offsets), figsize=(6.0 * len(offsets), 5.5))
    for index, (ax, offset) in enumerate(zip(axes, offsets)):
        dm = round(stream_dm + offset, 2)
        result = generic_search_single_dm(distance_modulus=dm, isochrone=ISOCHRONE, search=search,
                                          smoothing_deg=SMOOTHING_DEG)
        vmin, vmax = zoom_colour_scale(result, lon_range, lat_range)
        with_track = index == len(offsets) - 1
        _, sky = plot_generic_search_residual_map(
            result, lon_range=lon_range, lat_range=lat_range, vmin=vmin, vmax=vmax, ax=ax,
            title=f"dm {format_distance_modulus(dm)}" + (", track" if with_track else ""))
        if with_track:
            track = tracks[row.stream_name]
            draw_stream_track_on_sky(sky, track["ra"], track["dec"], color=STATUS_COLORS[row.status], lw=1.2,
                                     ls="--", alpha=0.9)
    observed = "not filled" if pd.isna(row.observed) else row.observed
    fig.suptitle(f"{row.stream_name} ({row.track_name}): dm {row.dm_min:.2f} to {row.dm_max:.2f}, width "
                 f"{row.width_deg:.2f} deg, status {row.status}, observed {observed}")
    return fig

In [ ]:
%%time
streams_folder = OUTPUT_FOLDER / "streams"
streams_folder.mkdir(parents=True, exist_ok=True)
rows = known_streams if STREAMS_SHOWN is None else known_streams[known_streams["stream_name"].isin(STREAMS_SHOWN)]
for row in rows.itertuples():
    fig = plot_stream_zooms(row)
    fig.savefig(streams_folder / f"{row.stream_name}.png", dpi=FRAME_DPI)
    display(fig)
    plt.close(fig)